# Reading a satellite image properly

Four questions to answer before the image is used for anything.  Everything is read
with `rasterio`, which is GDAL with a friendlier interface — Colab already has it.
The file is a THEOS-2 scene over Bangkok.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/readcheck.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'readcheck'):
    sys.modules.pop(m, None)
import rasterio
from readcheck import *

PATH = urllib.request.urlretrieve(f'{BASE}/dataset/intake/input.tif', 'input.tif')[0]

## Q1.  Am I reading it correctly?

Open it, read it, and look at what actually came back.

In [ ]:
with rasterio.open(PATH) as d:
    img = d.read()                     # (band, row, col)

print(img.shape, img.dtype)
print('values', img.min(), '-', img.max())

The array arrived and the values run 1 - 4095.  That is still not proof — two things
go wrong after this point.

In [ ]:
q1_read(PATH)

**Panel 1** — the numbers drawn straight onto a 0 - 65,535 scale, which is what
`uint16` suggests.  Black.  Nothing is broken; the scale is wrong.

**Panel 3** — the same pixels with the bands taken in the opposite order.  Water and
roofs come out blue.  It looks like a real image, which is what makes it dangerous.

## Q2.  How are the bands arranged?

Ask the file first.

In [ ]:
with rasterio.open(PATH) as d:
    print('bands      ', d.count, d.indexes)
    print('names      ', d.descriptions)
    print('colorinterp', tuple(c.name for c in d.colorinterp))

No names, and `colorinterp` says `gray` then `undefined` three times.  **The file does
not know which band is which** — that has to come from the product spec, or from
looking.

In [ ]:
q2_bands(PATH)

In the fourth band the river goes black and the parks go bright.  That is what water
and vegetation do to near infrared, so band 4 is NIR and bands 1 - 3 are R, G, B.

## Q3.  How big is it?

In [ ]:
with rasterio.open(PATH) as d:
    print('pixels', d.width, 'x', d.height, ' bands', d.count)
    print('pixel size', d.res)
    print('bounds    ', tuple(round(v, 1) for v in d.bounds), d.crs)

In [ ]:
q3_size(PATH)

Two different numbers answer this question.  How much ground it covers decides whether
it is the right scene; how much memory it takes decides whether it opens at all.

## Q4.  What type are the numbers?

In [ ]:
import numpy as np

with rasterio.open(PATH) as d:
    print('dtype', d.dtypes[0])
print('the box holds 0 -', np.iinfo(img.dtype).max)
print('the data uses 0 -', img.max())

In [ ]:
q4_type(PATH)

`uint16` is the size of the box, not the amount in it.  This sensor is 12-bit, so 94 %
of the box is never used — and assuming otherwise is what produced the black picture
in Q1.